# Atomic X-ray scattering factors and effective electron densities vs energy

This notebook demonstrates the **updated `atomic-scattering-factors` package (v0.3.0 API)**:

1. Plot the real and imaginary atomic forward-scattering factors, $f_1(E)$ and $f_2(E)$.
2. Plot the complex effective electron densities of Au, water, and SiO₂.
3. Compare the physically coherent density magnitude $|\tilde\rho_e(E)|$ with the package's **legacy** real-valued output.
4. (Optional) Plot complex particle–solvent contrast and the energy dependence of $|\Delta\tilde\rho_e|^2$ as a simple ASAXS illustration.

**Conventions:** photon energies are passed to the package in **eV**; plot axes use **keV**. The package returns densities in electrons/m³; we plot **electrons/Å³** ($1\ \mathrm{electron/\AA^3}=10^{30}\ \mathrm{electrons/m^3}$).

This is a *forward-scattering / small-$q$ approximation*. The atomic scattering factors are not chemical-state-sensitive near-edge measurements. No absorption, energy-bandwidth, structure-factor, or detector corrections are included in the illustrative intensity ratio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from atomic_scattering_factors import get_scattering_factors, effective_electron_density

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.25})

## 1. Choose energies and materials

The default range includes the **Au L₃ absorption edge** near 11.919 keV. Change the energy interval, list of elements, or material densities to suit your experiment. The material densities below are **illustrative bulk values**; replace them with the densities appropriate for your samples.

In [ ]:
# Only change this cell to customize the plots.
E_MIN_KEV, E_MAX_KEV = 8.0, 14.0
N_POINTS = 1501
AU_L3_KEV = 11.919  # approximate reference edge energy

ELEMENTS = ["Au", "Si", "O"]
# Material label -> (chemical formula, density in g/cm³)
MATERIALS = {
    "Gold": ("Au", 19.32),
    "Water": ("H2O", 1.00),
    "Silica": ("SiO2", 2.20),
}

energy_keV = np.linspace(E_MIN_KEV, E_MAX_KEV, N_POINTS)
energy_eV = energy_keV * 1e3
ELECTRONS_PER_M3_TO_PER_A3 = 1e-30

print(f"Photon energies: {E_MIN_KEV:.3f}–{E_MAX_KEV:.3f} keV ({N_POINTS} points)")

## 2. Atomic scattering factors, $f_1(E)$ and $f_2(E)$

These values come from the scattering-factor backend used by the installed `atomic-scattering-factors` package. Both factors are real-valued arrays; together they define the complex amplitude $f(E)=f_1(E)+i f_2(E)$.

In [ ]:
atomic_factors = {}
for element in ELEMENTS:
    f1, f2 = get_scattering_factors(element, energy_eV)
    atomic_factors[element] = (np.asarray(f1), np.asarray(f2))

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, constrained_layout=True)
for element, (f1, f2) in atomic_factors.items():
    axes[0].plot(energy_keV, f1, label=element)
    axes[1].plot(energy_keV, f2, label=element)
axes[0].set_ylabel(r"$f_1(E)$ (electrons)")
axes[1].set_ylabel(r"$f_2(E)$ (electrons)")
axes[1].set_xlabel("Photon energy (keV)")
for ax in axes:
    if E_MIN_KEV <= AU_L3_KEV <= E_MAX_KEV and "Au" in ELEMENTS:
        ax.axvline(AU_L3_KEV, color="0.45", lw=1, ls="--", label="Au L₃ (approx.)")
    ax.legend(ncol=2)
plt.show()

### Zoom into the Au L₃ edge

Close to an absorption edge, interpolation between tabulated points can matter. The curves below are interpolated reference values, not an experimentally measured XANES spectrum.

In [ ]:
if "Au" in atomic_factors:
    f1_au, f2_au = atomic_factors["Au"]
    zoom = (energy_keV >= AU_L3_KEV - 0.12) & (energy_keV <= AU_L3_KEV + 0.12)
    fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True, constrained_layout=True)
    axes[0].plot(energy_keV[zoom], f1_au[zoom], label="Au f₁")
    axes[1].plot(energy_keV[zoom], f2_au[zoom], label="Au f₂")
    for ax in axes:
        ax.axvline(AU_L3_KEV, color="0.45", lw=1, ls="--")
        ax.legend()
    axes[0].set_ylabel(r"$f_1$ (electrons)")
    axes[1].set_ylabel(r"$f_2$ (electrons)")
    axes[1].set_xlabel("Photon energy (keV)")
    plt.show()
else:
    print("Add 'Au' to ELEMENTS to show the Au L₃ detail.")

## 3. Complex effective electron density

For a material with formula-unit number density $n$ and stoichiometric coefficients $N_j$:

$$
\tilde\rho_e(E) = n\sum_j N_j\big[f_{1,j}(E)+i f_{2,j}(E)\big].
$$

We plot $\mathrm{Re}\,\tilde\rho_e$, $\mathrm{Im}\,\tilde\rho_e$, and $|\tilde\rho_e|$. Use `complex=True`; the default `complex=False` deliberately retains the legacy definition for backward compatibility.

In [ ]:
complex_densities = {}
legacy_densities = {}
for label, (formula, mass_density) in MATERIALS.items():
    kwargs = dict(sample_str=formula, mass_density_g_per_cm3=mass_density, energies_eV=energy_eV)
    complex_densities[label] = np.asarray(effective_electron_density(**kwargs, complex=True)) * ELECTRONS_PER_M3_TO_PER_A3
    legacy_densities[label] = np.asarray(effective_electron_density(**kwargs, complex=False)) * ELECTRONS_PER_M3_TO_PER_A3

fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True, constrained_layout=True)
for label, rho in complex_densities.items():
    axes[0].plot(energy_keV, rho.real, label=label)
    axes[1].plot(energy_keV, rho.imag, label=label)
    axes[2].plot(energy_keV, np.abs(rho), label=label)
axes[0].set_ylabel(r"$\mathrm{Re}(\tilde\rho_e)$ (e/Å³)")
axes[1].set_ylabel(r"$\mathrm{Im}(\tilde\rho_e)$ (e/Å³)")
axes[2].set_ylabel(r"$|\tilde\rho_e|$ (e/Å³)")
axes[2].set_xlabel("Photon energy (keV)")
for ax in axes:
    if E_MIN_KEV <= AU_L3_KEV <= E_MAX_KEV:
        ax.axvline(AU_L3_KEV, color="0.45", lw=1, ls="--")
    ax.legend()
plt.show()

## 4. Legacy density vs the magnitude of the coherent complex density

For compounds, the **legacy** definition and the physically coherent magnitude are generally not identical:

$$
\rho_{\mathrm{legacy}}(E)=n\sum_j N_j |f_j(E)|,
\qquad
|\tilde\rho_e(E)|=n\left|\sum_j N_j f_j(E)\right|.
$$

They agree for a pure element (up to roundoff), but need not agree for a compound. Neither expression should be confused with the **real part** of the coherent density.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for label in MATERIALS:
    legacy = legacy_densities[label]
    coherent_abs = np.abs(complex_densities[label])
    relative_percent = 100 * (legacy / coherent_abs - 1)
    ax.plot(energy_keV, relative_percent, label=label)
    print(f"{label:>8s}: maximum |legacy / |complex| - 1| = {np.max(np.abs(relative_percent)):.5g} %")
ax.axhline(0, color="0.5", lw=1)
ax.set(xlabel="Photon energy (keV)", ylabel=r"$(\rho_{\rm legacy}/|\tilde\rho_e|-1)\times100$ (%)",
       title="Difference between the legacy and coherent density magnitudes")
ax.legend()
plt.show()

## 5. Example: ASAXS contrast between gold and water

For a homogeneous particle in a solvent, the (complex) electron-density contrast is

$$\Delta\tilde\rho_e(E)=\tilde\rho_{e,\mathrm{particle}}(E)-\tilde\rho_{e,\mathrm{solvent}}(E).$$

At fixed geometry and neglecting other effects, the particle scattering contribution scales as $|\Delta\tilde\rho_e(E)|^2$. Here it is **normalized to the first energy point**, so this is *not* an absolute measured SAXS intensity curve.

In [ ]:
if {"Gold", "Water"}.issubset(complex_densities):
    delta = complex_densities["Gold"] - complex_densities["Water"]
    relative_intensity = np.abs(delta)**2 / np.abs(delta[0])**2
    fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, constrained_layout=True)
    axes[0].plot(energy_keV, delta.real, label=r"Re $\Delta\tilde\rho_e$")
    axes[0].plot(energy_keV, delta.imag, label=r"Im $\Delta\tilde\rho_e$")
    axes[0].plot(energy_keV, np.abs(delta), label=r"$|\Delta\tilde\rho_e|$")
    axes[0].set_ylabel("Contrast (e/Å³)")
    axes[0].legend()
    axes[1].plot(energy_keV, relative_intensity, label=r"$|\Delta\tilde\rho_e(E)|^2/|\Delta\tilde\rho_e(E_0)|^2$")
    axes[1].set(xlabel="Photon energy (keV)", ylabel="Relative contrast squared")
    axes[1].legend()
    for ax in axes:
        if E_MIN_KEV <= AU_L3_KEV <= E_MAX_KEV:
            ax.axvline(AU_L3_KEV, color="0.45", lw=1, ls="--")
    plt.show()
else:
    print("Add 'Gold' and 'Water' to MATERIALS to run this example.")

## 6. Values at individual photon energies

Use this short cell when you need a numerical result instead of a plot. It prints atomic $f_1$, $f_2$ for Au and the real, imaginary, and absolute complex electron density for gold and water. Adjust `E_SELECTED_KEV` as needed.

In [ ]:
E_SELECTED_KEV = [8.0, 10.0, 11.85, 11.90, 11.93, 12.0, 13.0]
print("E (keV)   Au f1       Au f2       Re rho_Au      Im rho_Au      |rho_Au|      |rho_H2O|")
print(" " * 31 + "(electron densities in e/Å³)")
for E in E_SELECTED_KEV:
    f1, f2 = get_scattering_factors("Au", E * 1000)
    rho_au = effective_electron_density("Au", 19.32, E * 1000, complex=True) * 1e-30
    rho_water = effective_electron_density("H2O", 1.00, E * 1000, complex=True) * 1e-30
    print(f"{E:7.3f}  {f1:9.4f}   {f2:9.4f}     {rho_au.real:10.5f}     {rho_au.imag:10.5f}     {abs(rho_au):10.5f}    {abs(rho_water):10.5f}")

### Notes

- The package's `get_scattering_factors()` expects **eV**, not keV.
- Complex density comes from `effective_electron_density(..., complex=True)`; `np.abs(...)` calculates its modulus.
- Material mass density is in **g/cm³**, while the function returns **electrons/m³**. For **electrons/nm³**, multiply by `1e-27` instead of `1e-30`.
- The imaginary part is essential near absorption edges. For actual quantitative ASAXS, measure or calibrate the resonant terms where possible and include experimental energy bandwidth, absorption, and fluorescence effects as applicable.